In [ ]:
!wget https://www.lamsade.dauphine.fr/~cazenave/project2025.zip
!unzip project2025.zip
!ls -l

!pip uninstall -y tensorflow
!pip install tensorrt-bindings==8.6.1
!pip install --extra-index-url https://pypi.nvidia.com tensorrt-libs
!pip install tensorflow[and-cuda]==2.15.0

# redemarrer la session

In [ ]:
# Import Libraries

import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
from tensorflow.keras import layers
from tensorflow.keras import regularizers
import gc
import golois

print ("Tensorflow version", tf.__version__)

In [ ]:
# Configuration

planes = 31
moves = 361
N = 10000
batch = 128


input_data = np.random.randint(2, size=(N, 19, 19, planes))
input_data = input_data.astype ('float32')

policy = np.random.randint(moves, size=(N,))
policy = keras.utils.to_categorical (policy)

value = np.random.randint(2, size=(N,))
value = value.astype ('float32')

end = np.random.randint(2, size=(N, 19, 19, 2))
end = end.astype ('float32')

groups = np.zeros((N, 19, 19, 1))
groups = groups.astype ('float32')

In [ ]:
# Get Validation Data

print ("getValidation", flush = True)
golois.getValidation (input_data, policy, value, end)

In [ ]:
# Model Creation
#
# Architecture simple de type AlphaGo-style, avec deux têtes :
# une policy (pour prédire un coup) et une value (pour prédire l’issue de la partie).
#
# Policy - Prédit le prochain coup. Valeur entre 1 et 361
#          Utilise la Categorical Cross Entropy.
# Value  - Prédit la probabilité de victoire. Valeur entre 0 et 1


#filters = 42
#filter_size = 3
#block_iteration=5

filters = 16
filter_size = 3
block_iteration=2


l2_reg = 0.0001


def get_iteration_block(x, filters=filters, filter_size=filter_size):
  x = layers.Conv2D(filters, filter_size, activation='relu', padding='same')(x)
  x = layers.Conv2D(filters*2, filter_size, activation='relu', padding='same')(x)
  x = layers.Conv2D(filters*4, filter_size, activation='relu', padding='same')(x)
  x = layers.Conv2D(filters*6, filter_size, activation='relu', padding='same')(x)
  return x

def get_model(filters=filters, filter_size=filter_size, l2_reg = l2_reg):

  input = keras.Input(shape=(19, 19, planes), name='board')
  x = layers.Conv2D(filters, 1, activation='relu', padding='same')(input)

  x = get_iteration_block(x, filters, filter_size)


  policy_head = layers.Conv2D(1, 1, activation='relu', padding='same', use_bias = False, kernel_regularizer=regularizers.l2(l2_reg))(x)
  policy_head = layers.Flatten()(policy_head)
  policy_head = layers.Activation('softmax', name='policy')(policy_head)

  value_head = layers.Conv2D(1, 1, activation='relu', padding='same', use_bias = False, kernel_regularizer=regularizers.l2(l2_reg))(x)
  value_head = layers.Flatten()(value_head)
  value_head = layers.Dense(50, activation='relu', kernel_regularizer=regularizers.l2(l2_reg))(value_head)
  value_head = layers.Dense(1, activation='sigmoid', name='value', kernel_regularizer=regularizers.l2(l2_reg))(value_head)

  model = keras.Model(inputs=input, outputs=[policy_head, value_head])
  return model

model = get_model(filters, filter_size, l2_reg)
model.summary ()


In [ ]:
# Model Compilation and Train

# Epochs number
epochs = 100

# Stochastic Gradient Descent (SGD) avec un taux d’apprentissage et un momentum personnalisés
learning_rate=0.01
momentum=0.9
optimizer = keras.optimizers.SGD(learning_rate=learning_rate, momentum=momentum)

# loss_weights : pondère la contribution des deux sorties à la loss totale
# - policy_weight : importance de la politique (jouer la bonne case)
# - value_weight : importance de la valeur (prédire le gagnant)
policy_weight = 1.0
value_weight = 100.0

model.compile(optimizer=optimizer,
              loss={'policy': 'categorical_crossentropy', 'value': 'binary_crossentropy'},
              loss_weights={'policy' : policy_weight, 'value' : value_weight},
              metrics={'policy': 'categorical_accuracy', 'value': 'mse'})

for i in range (1, epochs + 1):
    print ('epoch ' + str (i))

    # Met à jour lot de données d’entraînement, probablement de manière dynamique
    golois.getBatch (input_data, policy, value, end, groups, i * N)

    history = model.fit(input_data,
                        {'policy': policy, 'value': value},
                        epochs=1, batch_size=batch)
    if (i % 5 == 0):
        gc.collect ()
    if (i % epochs == 0):
        golois.getValidation (input_data, policy, value, end)
        val = model.evaluate (input_data,
                              [policy, value], verbose = 0, batch_size=batch)
        print ("val =", val)
        model.save ('malikchettih.h5')
